# 04 — Model comparison

**Target.** Δvolume between Skogsstyrelsen inventory cycles, `y = volym_omdrev2 − volym_omdrev1` (m³/ha), restricted to `is_stable_forest` pixels.

**What this notebook does.** Compares eight regression models — a frequentist linear baseline, three Pyro-SVI Bayesian models (BLR, hierarchical, BNN), exact GP and SVGP, plus Random Forest and histogram gradient boosting — on point accuracy, probabilistic calibration, runtime, memory, and how each scales with the training set.

**Spatial split.** 20 % of BK indexruta (500 m × 500 m) cells are held out, fixed across every model and every scaling step. Train sizes are nested: increasing N adds BK cells from a fixed deterministic ordering. A `random_split` row at full N quantifies the spatial-leakage premium that group-splits forfeit.

**Reproducibility.** Splits live in `cache/splits.json`, per-(model, scaling-step) results in `cache/results/`. With a warm cache the notebook re-runs in <1 minute; deleting `cache/results/` triggers a full re-train.

In [ ]:
from __future__ import annotations

import json
import pickle
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from tabulate import tabulate

import notebook_helpers as H
from notebook_helpers import (
    SEED, FEATURE_COLS, FORBIDDEN_FEATURES, TARGET_COL,
    CACHE_DIR, SPLITS_JSON, DATA_PARQUET,
)
from models import MODELS

warnings.filterwarnings('ignore', category=UserWarning)
np.random.seed(SEED)
torch.manual_seed(SEED)

CSV_PATH = Path('out_10km_idx_preprocessed.csv')
print(f'Project dir : {Path.cwd()}')
print(f'CSV path    : {CSV_PATH}  exists={CSV_PATH.exists()}')
print(f'Cache dir   : {CACHE_DIR}')
print(f'Feature cols: {len(FEATURE_COLS)}  → {FEATURE_COLS}')

## 1. Setup and data loading

We read the preprocessed CSV (351 MB, 641,601 rows on a 12.5 m grid), apply `is_stable_forest` to drop non-forest pixels, lakes, and any pixel that lost height/biomass/volume between cycles, then construct the target Δvolume column.

In [ ]:
if not CSV_PATH.exists():
    import urllib.request
    url = ('https://github.com/Somon8/mbml-forest-pipeline/'
           'releases/download/v1.2-data/out_10km_idx_preprocessed.csv')
    print(f'CSV missing — downloading from {url} ...')
    urllib.request.urlretrieve(url, CSV_PATH)
    print(f'  → wrote {CSV_PATH} ({CSV_PATH.stat().st_size/1e6:.0f} MB)')

df_raw = H.load_dataset(CSV_PATH)
print(f'Raw rows : {len(df_raw):,}  cols : {len(df_raw.columns)}')

In [ ]:
df = H.apply_filter_and_features(df_raw)

# Acceptance check — no forbidden columns leak in
H.assert_no_forbidden(FEATURE_COLS)
print('Forbidden-feature check passed.')

print(f"y stats: mean={df[TARGET_COL].mean():.2f}  "
      f"std={df[TARGET_COL].std():.2f}  "
      f"p5={df[TARGET_COL].quantile(0.05):.2f}  "
      f"p95={df[TARGET_COL].quantile(0.95):.2f}")

## 2. Train / test split and scaling-axis construction

Split is **by BK indexruta cell** so test pixels never appear in any training subset; spatial autocorrelation makes pixel-level splits dramatically optimistic. Smaller scaling subsets are nested inside larger ones (deterministic shuffle, seed 42), so the scaling curves are monotonic in data, not subsample variance.

We persist the full split definition to `cache/splits.json` so the experiment is reproducible across reruns.

In [ ]:
if SPLITS_JSON.exists():
    splits = H.load_splits()
    print('Loaded splits from cache.')
else:
    splits = H.build_splits(df, n_cells_grid=(5, 25, 100, 250, 'all'),
                            test_frac=0.20, seed=SEED)
    H.save_splits(splits)
    print('Built and saved splits.')

n_test_bk = len(splits.test_bk)
n_pool_bk = len(splits.train_bk_ordered)
print(f'Test BK cells : {n_test_bk}')
print(f'Train pool BK : {n_pool_bk}')
print(f'Scaling grid  : {splits.n_cells_grid}')

for n in splits.n_cells_grid:
    train_bk = H.get_train_bk(splits, n)
    n_pix = df[df['BK'].isin(set(train_bk))].shape[0]
    print(f'  n_cells={str(n):>4}  → {len(train_bk):3d} BK cells, '
          f'{n_pix:>7,} pixels')

In [ ]:
# Sanity: verify nesting (smaller subsets ⊂ larger).
subsets = [set(H.get_train_bk(splits, n)) for n in splits.n_cells_grid]
for i in range(len(subsets) - 1):
    assert subsets[i].issubset(subsets[i + 1]), \
        f'subset {i} not nested inside {i+1}'
print('Nesting check: smaller scaling subsets ⊂ larger. ✓')

# Sanity: train pool and test set are disjoint.
test_set = set(splits.test_bk)
for n, ss in zip(splits.n_cells_grid, subsets):
    assert not (ss & test_set), \
        f'training subset for n_cells={n} overlaps test BK'
print('Disjointness check: every train subset ∩ test = ∅. ✓')

In [ ]:
# Map of test BK cells overlaid on AOI extent.
fig, ax = plt.subplots(1, 1, figsize=(7, 7))
all_bk = df.groupby('BK')[['x', 'y']].mean()
is_test = all_bk.index.isin(splits.test_bk)
ax.scatter(all_bk.loc[~is_test, 'x'], all_bk.loc[~is_test, 'y'],
           s=8, c='#888', label=f'train pool ({(~is_test).sum()} BK)')
ax.scatter(all_bk.loc[is_test, 'x'],  all_bk.loc[is_test, 'y'],
           s=20, c='C3', marker='s',
           label=f'test ({is_test.sum()} BK)')
ax.set_aspect('equal')
ax.set_xlabel('Easting (m, EPSG:3006)')
ax.set_ylabel('Northing (m, EPSG:3006)')
ax.set_title('BK indexruta cells — held-out test split (20 %)')
ax.legend(loc='upper right', fontsize=9)
fig.tight_layout()
plt.show()

## 3. Feature inspection

Quick correlation heatmap of the standardized features on a 5k-row sub-sample. The cycle-1 forest-state block (volume / biomass / basal area / heights) is highly correlated within itself — that redundancy is documented in `PROJECT_JOURNAL.md §2.8` and is part of why probabilistic shrinkage matters for these covariates.

In [ ]:
df_train_full = df[df['BK'].isin(set(splits.train_bk_ordered))].copy()
X_full, y_full, g_full, _ = H.matrices(df_train_full)
mu_x, sigma_x = H.fit_scaler(X_full)
scaler = {'mu': mu_x, 'sigma': sigma_x, 'features': list(FEATURE_COLS)}
with open(CACHE_DIR / 'scaler.json', 'w') as f:
    json.dump({'mu': mu_x.tolist(), 'sigma': sigma_x.tolist(),
               'features': scaler['features']}, f, indent=2)
print(f'Scaler fit on full train pool (n={len(X_full):,}).')
print(pd.DataFrame({'feature': FEATURE_COLS,
                    'mean': mu_x, 'std': sigma_x}).round(3))

In [ ]:
# Correlation heatmap (5k-row subsample, post-standardization).
rng = np.random.default_rng(SEED)
idx = rng.choice(len(X_full), size=min(5000, len(X_full)),
                 replace=False)
Xs = H.apply_scaler(X_full[idx], mu_x, sigma_x)
C = np.corrcoef(Xs.T)

fig, ax = plt.subplots(figsize=(8, 6.5))
im = ax.imshow(C, vmin=-1, vmax=1, cmap='RdBu_r')
ax.set_xticks(range(len(FEATURE_COLS)))
ax.set_yticks(range(len(FEATURE_COLS)))
ax.set_xticklabels(FEATURE_COLS, rotation=70, ha='right', fontsize=8)
ax.set_yticklabels(FEATURE_COLS, fontsize=8)
ax.set_title('Feature correlation (standardized, 5k random rows)')
plt.colorbar(im, ax=ax, label='Pearson ρ')
fig.tight_layout()
plt.show()

## 4. Building the data + scaler cache

We persist the filtered + feature-engineered table to `cache/data.parquet` and re-read it for every fit. The scaler is fit **once** on the full train pool; every smaller scaling step and the test set apply this same scaler — keeps point comparisons fair across N.

In [ ]:
if not DATA_PARQUET.exists():
    keep = ['BK', 'x', 'y', TARGET_COL] + FEATURE_COLS
    df[keep].to_parquet(DATA_PARQUET)
    print(f'Wrote {DATA_PARQUET}.')
else:
    print(f'{DATA_PARQUET} already exists.')
print(f'  size: {DATA_PARQUET.stat().st_size/1e6:.1f} MB')

## 5. Per-model demos at the smallest scaling step

Each model fits at `n_cells = 5` (≈ a few thousand pixels) and we print predictions vs truth as a sanity check. This is **not** the full experiment — that comes in §6 — but it gives an early look at each model individually.

In [ ]:
def slice_step(df_full, n_cells):
    df_train, df_test = H.split_frames(df_full, splits, n_cells)
    X_tr_raw, y_tr, g_tr, _ = H.matrices(df_train)
    X_te_raw, y_te, g_te, coords_te = H.matrices(df_test)
    X_tr = H.apply_scaler(X_tr_raw, mu_x, sigma_x)
    X_te = H.apply_scaler(X_te_raw, mu_x, sigma_x)
    return X_tr, y_tr, g_tr, X_te, y_te, g_te, coords_te

X_tr5, y_tr5, g_tr5, X_te, y_te, g_te, coords_te = slice_step(df, 5)
print(f'n_cells=5  train n={len(X_tr5):,}  test n={len(X_te):,}')

### 5.linear — linear

Linear regression — the speed/sanity baseline. No uncertainty; we use it as a free reference for RMSE.

In [ ]:
rec = H.run_one('linear', 'demo_n5',
                X_tr5, y_tr5, g_tr5,
                X_te, y_te, g_te, coords_te,
                n_cells_eff=5, n_train_bk=5, force=True)
print('Metrics:', rec['metrics'])

### 5.blr — blr

Bayesian Linear Regression with mean-field SVI. Same linear predictor as the frequentist baseline, but with Normal priors on weights and a learned σ — so we get a predictive distribution per pixel.

In [ ]:
rec = H.run_one('blr', 'demo_n5',
                X_tr5, y_tr5, g_tr5,
                X_te, y_te, g_te, coords_te,
                n_cells_eff=5, n_train_bk=5, force=True,
                model_kwargs={'n_steps': 1500})
print('Metrics:', rec['metrics'])

### 5.hierarchical — hierarchical

Hierarchical regression with random intercept per BK cell. Hyperprior on group mean / spread, fixed slopes for the features. New BK cells fall back to the hyperprior at predict time, which inflates predictive variance for unseen groups.

In [ ]:
rec = H.run_one('hierarchical', 'demo_n5',
                X_tr5, y_tr5, g_tr5,
                X_te, y_te, g_te, coords_te,
                n_cells_eff=5, n_train_bk=5, force=True,
                model_kwargs={'n_steps': 1500})
print('Metrics:', rec['metrics'])

### 5.exact_gp — exact_gp

Exact Gaussian Process with Matérn-5/2 ARD kernel. Only feasible at very small N — beyond ~10k training rows the kernel matrix becomes intractable. We deliberately skip it at larger scaling steps.

In [ ]:
rec = H.run_one('exact_gp', 'demo_n5',
                X_tr5, y_tr5, g_tr5,
                X_te, y_te, g_te, coords_te,
                n_cells_eff=5, n_train_bk=5, force=True,
                model_kwargs={'n_steps': 300})
print('Metrics:', rec['metrics'])

### 5.svgp — svgp

Sparse Variational GP — same kernel as exact GP but with ~512 inducing points (initialised by k-means on training inputs), trained with minibatch SVI. Designed to scale to 100k+ rows.

In [ ]:
rec = H.run_one('svgp', 'demo_n5',
                X_tr5, y_tr5, g_tr5,
                X_te, y_te, g_te, coords_te,
                n_cells_eff=5, n_train_bk=5, force=True,
                model_kwargs={'n_inducing': 256, 'n_steps': 1000,
                              'batch_size': 4096})
print('Metrics:', rec['metrics'])

### 5.bnn — bnn

Bayesian Neural Network: 2 hidden layers, 64 ReLU units, Normal priors on weights, learned output noise. Mean-field variational guide. The non-linear cousin of BLR.

In [ ]:
rec = H.run_one('bnn', 'demo_n5',
                X_tr5, y_tr5, g_tr5,
                X_te, y_te, g_te, coords_te,
                n_cells_eff=5, n_train_bk=5, force=True,
                model_kwargs={'n_steps': 2000, 'batch_size': 4096})
print('Metrics:', rec['metrics'])

### 5.rf — rf

Random Forest. Frequentist baseline. We treat the empirical std across trees as an uncertainty estimate so we can score it on the same probabilistic metrics — see the calibration plot for whether that proxy is well-calibrated.

In [ ]:
rec = H.run_one('rf', 'demo_n5',
                X_tr5, y_tr5, g_tr5,
                X_te, y_te, g_te, coords_te,
                n_cells_eff=5, n_train_bk=5, force=True,
                model_kwargs={'n_estimators': 200})
print('Metrics:', rec['metrics'])

### 5.gbm — gbm

Histogram gradient boosting. Frequentist baseline, point-only — probabilistic metrics show as `n/a`.

In [ ]:
rec = H.run_one('gbm', 'demo_n5',
                X_tr5, y_tr5, g_tr5,
                X_te, y_te, g_te, coords_te,
                n_cells_eff=5, n_train_bk=5, force=True,
                model_kwargs={'max_iter': 300})
print('Metrics:', rec['metrics'])

## 6. Full experiment loop

Iterate every model over the full scaling grid. **Caching is automatic** — anything in `cache/results/` is reused; deleting a specific pickle re-fits just that one cell.

Special cases:
- Exact GP runs only at `n_cells ∈ {5, 25}`. Other steps log a skip reason and don't fit.
- After the group-split sweep, every model also fits a `random_split` row at full N: a random 80/20 pixel split. The delta between random and group rows quantifies the spatial-leakage premium.

In [ ]:
MODEL_KWARGS = {
    'linear': {},
    'blr': {'n_steps': 2000},
    'hierarchical': {'n_steps': 2500},
    'exact_gp': {'n_steps': 500},
    'svgp': {'n_inducing': 512, 'n_steps': 2000, 'batch_size': 4096},
    'bnn': {'n_steps': 4000, 'batch_size': 4096},
    'rf': {'n_estimators': 200},
    'gbm': {'max_iter': 300},
}
EXACT_GP_MAX_BK = 25
SCALING_GRID = list(splits.n_cells_grid)
MODEL_ORDER = ['linear', 'blr', 'hierarchical', 'exact_gp',
               'svgp', 'bnn', 'rf', 'gbm']
print('Scaling grid :', SCALING_GRID)
print('Models       :', MODEL_ORDER)

In [ ]:
for n_cells in SCALING_GRID:
    print(f'\n=== n_cells = {n_cells} ===')
    Xtr, ytr, gtr, Xte, yte, gte, coords_te = slice_step(df, n_cells)
    n_bk = len(H.get_train_bk(splits, n_cells))
    print(f'    train pixels = {len(Xtr):,}  ({n_bk} BK cells)  '
          f'test pixels = {len(Xte):,}')
    label = f'group_n{n_cells}'
    for name in MODEL_ORDER:
        if name == 'exact_gp' and (
            n_cells == 'all' or int(n_cells) > EXACT_GP_MAX_BK):
            print(f'  skip exact_gp@{n_cells}: too large for an '
                  f'O(N^3) kernel matrix at N={len(Xtr):,}')
            continue
        H.run_one(name, label, Xtr, ytr, gtr, Xte, yte, gte, coords_te,
                  n_cells_eff=n_cells, n_train_bk=n_bk,
                  is_random_split=False,
                  model_kwargs=MODEL_KWARGS[name])
print('\nGroup-split scan complete.')

In [ ]:
# Random-split row: 80/20 pixel-level split at full N.
X_all_raw, y_all, g_all, coords_all = H.matrices(df)
rng = np.random.default_rng(SEED)
perm = rng.permutation(len(X_all_raw))
n_train = int(0.8 * len(X_all_raw))
tr_idx = perm[:n_train]
te_idx = perm[n_train:]
X_tr_rand = H.apply_scaler(X_all_raw[tr_idx], mu_x, sigma_x)
X_te_rand = H.apply_scaler(X_all_raw[te_idx], mu_x, sigma_x)
y_tr_rand, y_te_rand = y_all[tr_idx], y_all[te_idx]
g_tr_rand, g_te_rand = g_all[tr_idx], g_all[te_idx]
coords_te_rand = coords_all[te_idx]
print(f'random-split: train {len(X_tr_rand):,}  test {len(X_te_rand):,}')

for name in MODEL_ORDER:
    if name == 'exact_gp':
        print(f'  skip exact_gp@random_split: full N too large')
        continue
    H.run_one(name, 'random_split',
              X_tr_rand, y_tr_rand, g_tr_rand,
              X_te_rand, y_te_rand, g_te_rand, coords_te_rand,
              n_cells_eff='all', n_train_bk=None,
              is_random_split=True,
              model_kwargs=MODEL_KWARGS[name])
print('\nRandom-split sweep complete.')

## 7. Aggregated results table

Pivot of all cached runs. The probabilistic columns (`nlpd`, `crps`, `coverage_90`) read as NaN for non-probabilistic models — expected, those are noted as `n/a` in the report.

In [ ]:
results = H.aggregate_results()
print(f'{len(results)} cached runs')
results = results.sort_values(['model', 'is_random_split', 'n_cells_eff'])
results.head(20)

In [ ]:
# Pivot: rows = model, cols = (n_cells, metric)
group_only = results[~results['is_random_split']].copy()
group_only['n_cells_eff'] = group_only['n_cells_eff'].astype(str)
metrics_to_show = ['rmse', 'r2', 'nlpd', 'crps', 'coverage_90',
                   'fit_seconds']
tidy = group_only.melt(
    id_vars=['model', 'n_cells_eff'],
    value_vars=metrics_to_show,
    var_name='metric', value_name='value')
pivot = tidy.pivot_table(index='model',
                         columns=['n_cells_eff', 'metric'],
                         values='value')
# Order columns by n_cells_eff numerically where possible.
n_order = [str(n) for n in SCALING_GRID]
n_order = [n for n in n_order if n in
           pivot.columns.get_level_values(0)]
pivot = pivot.reindex(columns=n_order, level=0)
pivot = pivot.reindex(index=[m for m in MODEL_ORDER
                              if m in pivot.index])

print('Group-split results — rows = model, columns = (n_cells, metric):')
print(tabulate(pivot.round(3), headers='keys',
               tablefmt='github', floatfmt='.3f'))

In [ ]:
rs = results[results['is_random_split']].copy()
if len(rs):
    print('\nRandom-split row (full N, 80/20 pixel split):')
    cols = ['model', 'n_train', 'rmse', 'r2', 'nlpd', 'crps',
            'coverage_90', 'fit_seconds', 'peak_memory_mb']
    print(tabulate(rs[cols].sort_values('model').round(3),
                   headers='keys', tablefmt='github',
                   showindex=False, floatfmt='.3f'))

## 8. Scaling figures

Three side-by-side log-x panels: RMSE / NLPD / fit time vs the scaling axis. The fit-time panel is the one that exposes which models scale gracefully — exact GP curves drop off the right edge where it stopped being feasible.

In [ ]:
def numeric_n(s):
    if s == 'all':
        # Resolve 'all' to its actual BK count for plotting.
        return len(splits.train_bk_ordered)
    return int(s)

scale_df = group_only.copy()
scale_df['n_cells_num'] = scale_df['n_cells_eff'].apply(numeric_n)

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5), sharex=True)
panels = [('rmse', 'RMSE  (m³/ha)'),
          ('nlpd', 'NLPD'),
          ('fit_seconds', 'Fit time (s)')]
for ax, (col, ylabel) in zip(axes, panels):
    for name in MODEL_ORDER:
        sub = scale_df[scale_df['model'] == name]
        if len(sub) == 0:
            continue
        sub = sub.sort_values('n_cells_num')
        v = sub[col].to_numpy()
        if np.isnan(v).all():
            continue
        ax.plot(sub['n_cells_num'], v, marker='o', label=name)
    ax.set_xscale('log')
    if col == 'fit_seconds':
        ax.set_yscale('log')
    ax.set_xlabel('n_cells (training BK indexrutas)')
    ax.set_ylabel(ylabel)
    ax.grid(True, which='both', alpha=0.3)
axes[0].set_title('Point accuracy: RMSE vs N')
axes[1].set_title('Probabilistic: NLPD vs N')
axes[2].set_title('Cost: fit time vs N')
axes[-1].legend(bbox_to_anchor=(1.02, 1), loc='upper left',
                fontsize=8)
fig.tight_layout()
plt.show()

## 9. Calibration

Empirical coverage at nominal levels α ∈ {0.1, 0.15, …, 0.95} for every probabilistic model at full N. A perfectly-calibrated model lies on the diagonal. Above-diagonal = under-confident (intervals too wide), below = over-confident.

In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 6.5))
ax.plot([0, 1], [0, 1], '--', c='k', lw=1, label='perfectly calibrated')
for name in MODEL_ORDER:
    label = 'group_nall'
    p = H.cache_path(name, label)
    if not p.exists():
        continue
    rec = H.load_record(name, label)
    if rec['alphas'] is None:
        continue
    ax.plot(rec['alphas'], rec['emp_coverage'],
            marker='o', label=name)
ax.set_xlabel('Nominal coverage α')
ax.set_ylabel('Empirical coverage on test')
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.set_title('Calibration at full N (group split)')
ax.legend(loc='lower right', fontsize=9)
ax.grid(True, alpha=0.3)
fig.tight_layout()
plt.show()

## 10. Spatial residual maps (full N)

One residual map per model at full-N group split, with a shared diverging colour scale. Spatially-coherent residuals — large blobs of one colour — indicate that the model is missing structure the other features don't capture (typically the tails of the spatial-autocorrelation signal that we deliberately did not exploit via spatial features).

In [ ]:
models_with_full = [m for m in MODEL_ORDER
                    if H.cache_path(m, 'group_nall').exists()]
if not models_with_full:
    print('No full-N records yet — run the experiment loop in §6.')
else:
    # Choose a fixed diverging scale: clip at p1/p99 of residuals.
    all_resid = []
    for name in models_with_full:
        rec = H.load_record(name, 'group_nall')
        all_resid.append(rec['mu'] - rec['y_test'])
    all_resid = np.concatenate(all_resid)
    vmax = float(np.quantile(np.abs(all_resid), 0.99))

    n = len(models_with_full)
    cols = 4
    rows = int(np.ceil(n / cols))
    fig, axes = plt.subplots(rows, cols,
                             figsize=(4 * cols, 4 * rows),
                             squeeze=False)
    for i, name in enumerate(models_with_full):
        ax = axes[i // cols, i % cols]
        rec = H.load_record(name, 'group_nall')
        x = rec['coords_test'][:, 0]
        y = rec['coords_test'][:, 1]
        r = rec['mu'] - rec['y_test']
        sc = ax.scatter(x, y, c=r, cmap='RdBu_r',
                        vmin=-vmax, vmax=vmax, s=1.5, marker='s')
        ax.set_aspect('equal')
        ax.set_title(f'{name}\nRMSE={rec["metrics"]["rmse"]:.1f}',
                     fontsize=10)
        ax.set_xticks([])
        ax.set_yticks([])
    for j in range(len(models_with_full), rows * cols):
        axes[j // cols, j % cols].axis('off')
    fig.suptitle('Spatial residuals (predicted − true Δvolume, '
                 'm³/ha) — full-N group split',
                 y=1.02, fontsize=12)
    cax = fig.add_axes([0.92, 0.15, 0.015, 0.7])
    fig.colorbar(sc, cax=cax, label='residual (m³/ha)')
    fig.tight_layout(rect=[0, 0, 0.91, 1.0])
    plt.show()

In [ ]:
# Residual histogram + QQ + residual-vs-predicted, per probabilistic
# model at full N. Diagnostic-grade.
from scipy import stats as sst
diag_models = [m for m in models_with_full
               if MODELS[m].is_probabilistic]
if diag_models:
    fig, axes = plt.subplots(len(diag_models), 3,
                             figsize=(13, 2.8 * len(diag_models)),
                             squeeze=False)
    for i, name in enumerate(diag_models):
        rec = H.load_record(name, 'group_nall')
        r = rec['mu'] - rec['y_test']
        ax_h, ax_q, ax_s = axes[i]
        ax_h.hist(r, bins=80)
        ax_h.set_title(f'{name}: residual histogram', fontsize=10)
        ax_h.set_xlabel('residual (m³/ha)')
        sst.probplot(r, dist='norm', plot=ax_q)
        ax_q.set_title(f'{name}: QQ vs Normal', fontsize=10)
        ax_s.scatter(rec['mu'], r, s=1, alpha=0.3)
        ax_s.axhline(0, c='k', lw=1)
        ax_s.set_xlabel('predicted Δvolume')
        ax_s.set_ylabel('residual (m³/ha)')
        ax_s.set_title(f'{name}: residual vs predicted', fontsize=10)
    fig.tight_layout()
    plt.show()

## 11. Discussion

### Headline findings

1. **Point accuracy.** Compare the RMSE column at `n_cells = all` in the §7 table. Tree-based models tend to win on raw RMSE; the Bayesian / GP models are typically within a few m³/ha but cost much more compute.
2. **Calibration.** The §9 plot is the most important figure for this course. Pyro mean-field Bayesian models (BLR / hier / BNN) should track close to the diagonal because their predictive σ explicitly includes likelihood noise. The Random Forest's tree-std proxy is a frequentist estimate of *model* variance only — it is typically over-confident.
3. **Speed.** Linear and GBM are fastest; SVGP has a high constant but better large-N scaling than the BNN; the exact GP is only viable up to ~25 BK cells. Fit-time scaling is the §8 right panel.
4. **Random vs group split.** The §7 random-split table should show substantially better metrics than the corresponding `group_nall` row for every model — the difference is the spatial leakage premium. If those numbers are similar, the spatial structure of `delta_volym` is weaker than expected (or a bug).

### Open items

- The hierarchical model only partially pools the **intercept**; a logical next step is partial pooling on slopes too, or layering the hierarchy by `Storruta`. The current design keeps the comparison clean against a single linear-in-features baseline.
- Spatial features (`*_omdrev1`-derived neighbourhood / directional / distance) were deliberately excluded to stay leakage-aware. Some of them (e.g. `dist_to_no_forest_m`) depend on `is_no_forest` which uses both cycles — see `PROJECT_JOURNAL.md` §6 for spatial-feature design notes.
- SLU Skogskarta layers may have been produced post-omdrev1 — they are kept in because species composition is biologically near-stationary on a decade scale, but the assumption is worth stating explicitly in the report.
- The cached results live in `cache/results/`; deleting individual pickles re-fits just those (model, n_cells) cells on the next run.